<a href="https://colab.research.google.com/github/monilkarajapaksha/Personal_Knowledge_Base_Chatbot/blob/main/Personal_Knowledge_Base_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-community langchain-chroma chromadb sentence-transformers anthropic

In [3]:
import os
from google.colab import files

os.makedirs("data", exist_ok=True)
uploaded = files.upload()  # pick your CVs / notes

for fname in uploaded:
    os.rename(fname, f"data/{fname}")

Saving Monilka_CV__AI (1).pdf to Monilka_CV__AI (1).pdf
Saving Monilka_Rajapaksha_CV_dxdy.pdf to Monilka_Rajapaksha_CV_dxdy.pdf
Saving Monilka_Rajapaksha_CV_PredictivAI.pdf to Monilka_Rajapaksha_CV_PredictivAI.pdf


In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, UnstructuredFileLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use UnstructuredFileLoader for broader file type support (like PDF, DOCX)
loader = DirectoryLoader("data", glob="**/*.*", loader_cls=UnstructuredFileLoader)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"{len(chunks)} chunks created")

/tmp/ipykernel_17696/4048960324.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader, UnstructuredFileLoader


160 chunks created


In [3]:
!pip install -q langchain-huggingface

In [4]:
!pip install -q -U protobuf

In [5]:

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
results = retriever.invoke("a question you know the answer to")
for doc in results:
    print(doc.metadata.get("source"), "->", doc.page_content[:150])

data/Monilka_Rajapaksha_CV_QE.pdf -> 2
data/Monilka_Rajapaksha_CV_AI.pdf -> EXPERIENCE

Mar 2026 – Present Velaris Sri Lanka Associate Quality Engineer Moratuwa, Sri Lanka • AI-Driven Automation: Building an AI-driven regressi
data/Monilka_Rajapaksha_PhD_Candidate_CV_RMIT.pdf -> Large Language Models (LLMs) and autonomous AI agents to accelerate, self-heal, and elevate software quality assurance.
data/Monilka_CV__AI (1).pdf -> retraining, latency optimization). Backed by a production quality-engineering background that brings rare rigor to building reliable, deployable ML.


In [7]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.1 MB/s eta 0:00:00


In [8]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))

def ask(query, k=4):
    docs = vectorstore.as_retriever(search_kwargs={"k": k}).invoke(query)
    context = "\n\n".join(f"[Source: {d.metadata.get('source')}]\n{d.page_content}" for d in docs)
    prompt = f"""Answer using ONLY the context below. If it's not there, say you don't know.

Context:
{context}

Question: {query}
Answer:"""
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    sources = sorted(set(d.metadata.get("source", "unknown") for d in docs))
    return resp.choices[0].message.content, sources

In [9]:
def ask_with_citations(query, k=4):
    answer, sources = ask(query, k=k)
    print(answer)
    print("\nSources:", ", ".join(sources))

In [11]:
ask_with_citations("something answerable from your documents")
ask_with_citations("something definitely NOT in your documents")  # should say "I don't know"

What company do I currently work for as an Associate Quality Engineer? 

Answer: Velaris Sri Lanka.

Sources: data/Job Application - Quality Assurance Engineer.pdf, data/Monilka_CV__AI (1).pdf, data/Monilka_CV__QE (3).pdf, data/Monilka_Rajapaksha_CV_Quality_Engineer.pdf
I cannot provide any information about that.

Sources: data/Monilka_Rajapaksha_CV_AI.pdf, data/Monilka_Rajapaksha_CV_Quality_Engineer.pdf, data/Monilka_Rajapaksha_CV_dxdy.pdf
